# TVLA Computation — Balanced Welch's $t$-Test

This notebook computes the **non-specific fixed-vs-random TVLA** (Test Vector
Leakage Assessment) on pre-recorded power traces stored in HDF5 format.

## Methodology

We implement a streaming (chunked) computation of per-sample means and
variances, then apply **Welch's unequal-variance $t$-test** at every sample
point.  The trace sets are balanced to $\min(N_\text{fixed}, N_\text{random})$
to avoid bias.

Two significance thresholds are reported:

| Criterion | Value | Reference |
|---|---|---|
| Standard | $\pm 4.5$ | Goodwill et al. (NIST, 2011) |
| Mini-p (Bonferroni) | Data-dependent | Controls FWER at $\alpha = 10^{-5}$ |

### References

- B. Goodwill et al., 
,
 NIST Non-Invasive Attack Testing Workshop, 2011.
- T. Schneider and A. Moradi, 
 CHES 2015.

---

## 1 — Configuration

All paths and statistical parameters are imported from the centralised
`src.config` module.  Override any value in this cell if needed for a
specific experiment.

In [ ]:
import sys, os

# Ensure the repository root is on the Python path
# Resolve repo root robustly (works regardless of kernel cwd)
_nb_dir = os.path.dirname(os.path.abspath("__file__"))
for _c in [os.path.join(_nb_dir, ".."), _nb_dir]:
    if os.path.isdir(os.path.join(os.path.abspath(_c), "src")):
        REPO_ROOT = os.path.abspath(_c); break
else:
    REPO_ROOT = os.path.abspath(".")
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from src.config import (
    WORKDIR, PLOTS_DIR, THRESHOLD_STD, ALPHA_GLOBAL, CHUNK_SIZE,
    DATA_COMBINED,
)

# ---------------------------------------------------------------------------
# Dataset selection — point to the combined HDF5 file to analyse
# ---------------------------------------------------------------------------
TRACE_FILE = os.path.join(DATA_COMBINED,
    "traces_combined_TVLA_v6_1M_smpl_7_5_Mhz.h5")

SAVE_PATH = os.path.join(PLOTS_DIR, "tvla_results_computed.npz")

# Set to 0 to use the full dataset, or to a positive integer for quick tests
SEL_NTRACES = 0

print(f"Trace file : {TRACE_FILE}")
print(f"Output     : {SAVE_PATH}")

## 2 — Streaming Welch's $t$-Test

Statistics are accumulated in a single pass over the HDF5 datasets using
the `calculate_stats_online` function from `src.tvla_core`, which computes
sums and sums-of-squares in chunks of `CHUNK_SIZE` traces to limit memory
usage.

In [ ]:
import h5py
import numpy as np

from src.tvla_core import calculate_stats_online, compute_threshold_minip

if not os.path.exists(TRACE_FILE):
    raise FileNotFoundError(f"Trace file not found: {TRACE_FILE}")

print("--- TVLA COMPUTATION START ---\n")

with h5py.File(TRACE_FILE, "r") as f:
    dset_fixed  = f["traces_fixed"]["traces"]
    dset_random = f["traces_random"]["traces"]

    len_fixed  = dset_fixed.shape[0]
    len_random = dset_random.shape[0]

    # --- Balanced trace count ---
    if SEL_NTRACES > 0:
        min_traces = SEL_NTRACES
        print(f"NOTE: SEL_NTRACES forced to {SEL_NTRACES}.")
    else:
        min_traces = min(len_fixed, len_random)

    print(f"Dataset statistics:")
    print(f"  Fixed traces  : {len_fixed:,}")
    print(f"  Random traces : {len_random:,}")
    print(f"  Balanced N    : {min_traces:,}")
    print(f"  Samples/trace : {dset_fixed.shape[1]:,}\n")

    # --- Streaming mean / variance computation ---
    n_f, mean_f, var_f = calculate_stats_online(
        dset_fixed,  n_limit=min_traces, chunk_size=CHUNK_SIZE, desc="Fixed ")
    n_r, mean_r, var_r = calculate_stats_online(
        dset_random, n_limit=min_traces, chunk_size=CHUNK_SIZE, desc="Random")

# --- Welch's t-statistic ---
print("\nComputing Welch's t-statistic...")
numerator   = mean_f - mean_r
denominator = np.sqrt((var_f / n_f) + (var_r / n_r))
t_val = np.divide(numerator, denominator,
                  out=np.zeros_like(numerator), where=denominator != 0)
t_val = np.nan_to_num(t_val)

# --- Mini-p (Bonferroni) threshold ---
n_samples      = len(t_val)
threshold_minip = compute_threshold_minip(n_samples, ALPHA_GLOBAL)

print(f"\n--- RESULTS ---")
print(f"Max |t|             : {np.max(np.abs(t_val)):.2f}")
print(f"Standard threshold  : +/- {THRESHOLD_STD}")
print(f"Mini-p threshold    : +/- {threshold_minip:.4f}")

## 3 — Persist Results

The $t$-statistic vector and both thresholds are saved to a compressed
NumPy archive so that the plotting notebook can be run independently
without re-processing the full trace set.

In [ ]:
print(f"Saving to: {SAVE_PATH}")
np.savez(SAVE_PATH,
         t_val=t_val,
         threshold_minip=threshold_minip,
         threshold_std=THRESHOLD_STD,
         n_traces=min_traces)
print("Computation complete.")